# Day 01 Tutorial — Spark & PySpark Fundamentals

**Goal:** Understand Spark architecture and basic DataFrame operations.

## Learning objectives
- Driver, Executors, Cluster Manager
- Transformations vs Actions + lazy evaluation
- Create SparkSession and explore a DataFrame


### Environment setup
Skip pip install on Databricks/Fabric. Locally you may need: `pip install pyspark pandas`.


In [ ]:
# %pip install pyspark==3.5.1 pandas -q


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName('AzureDE-InterviewPrep')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark


## 1. Architecture (interview mental model)

| Component | Role |
|-----------|------|
| **Driver** | Runs main code, builds DAG, schedules tasks |
| **Executor** | Runs tasks and stores data |
| **Cluster Manager** | Allocates resources |

**Lazy evaluation:** Transformations build a plan; Actions trigger execution.


## 2. Create a DataFrame


In [ ]:
data = [
    (1, 'Alice', 'Sales', 70000),
    (2, 'Bob', 'Engineering', 90000),
    (3, 'Carol', 'Sales', 72000),
    (4, 'Dan', 'Engineering', 95000),
    (5, 'Eve', 'HR', 65000),
]
cols = ['emp_id', 'name', 'department', 'salary']
df = spark.createDataFrame(data, cols)
df.printSchema()
df.show()


## 3. Transformations (lazy) vs Actions (eager)


In [ ]:
high_paid = (
    df.filter(F.col('salary') > 70000)
      .select('name', 'department', 'salary')
      .withColumn('salary_k', F.round(F.col('salary') / 1000, 1))
)
print('Plan built; nothing executed yet.')
high_paid.explain(True)
high_paid.show()
print('Count:', high_paid.count())


## 4. Common operations


In [ ]:
(
    df.select(F.col('name').alias('employee_name'), 'department', 'salary')
      .filter(F.col('department') == 'Sales')
      .show()
)

df.withColumn(
    'band',
    F.when(F.col('salary') >= 90000, 'A')
     .when(F.col('salary') >= 70000, 'B')
     .otherwise('C'),
).show()


## Interview talking points
1. DataFrames have schema + Catalyst optimizer (prefer over RDDs for analytics).
2. `show` / `count` / `collect` are actions; avoid `collect` on large data.
3. Lazy evaluation chains transforms into one optimized job at action time.


## Mini practice


In [ ]:
# TODO: Engineering employees with salary_k and band
# Attempt first, then check solution cell


In [ ]:
(
    df.filter(F.col('department') == 'Engineering')
      .withColumn('salary_k', F.round(F.col('salary') / 1000, 1))
      .withColumn('band', F.when(F.col('salary') >= 90000, 'A').otherwise('B'))
      .select('name', 'salary', 'salary_k', 'band')
      .show()
)
